# Automated Climate Finance Policy Gap Analyser
**Question answered:** *Does our internal documentation cover what external climate-finance regulations and standards expect, and where are the gaps?*

**Pipeline:** internal PDFs → clause-level chunks → FAISS retrieval → rule-based evidence check (optional LLM refinement) → gap register (Excel/CSV) → PDF report.

**Inputs (two folders/files, easy to swap for real material)**
| Organization's User Input | Sample Gap Analysis Uploads | Can be Replaced |
|---|---|---|
| `sample_policies/*.pdf` | 7 fictional bank documents with deliberate gaps | Your real policy PDFs (text-based, with numbered section headings such as `3.1 Scenario analysis`) |
| `requirements_library.csv` | 27 paraphrased requirements across IFRS S2, CBK, Basel, ICMA GBP, EU CSRD/ESRS | A library you maintain from primary sources (one row per atomic requirement) |

**How a requirement is judged.** Each requirement lists the concepts a policy must contain (for example *scenario analysis* AND *a resilience or temperature pathway*); the first concept is the anchor and must be evidenced. The engine retrieves the best-matching clauses, checks which concepts appear, then downgrades status when the wording is aspirational (*will develop*, *under development*) or negated (*not yet implemented*, *not quantified*). It also flags stale references (for example TCFD-only). This baseline is transparent and repeatable; an LLM can then refine borderline cases when a key is available.

> The requirement texts are paraphrased summaries for screening, not legal text. Confirm status, scope and dates at the primary source before acting.

**Local ZIP upload:** The archive should contain one or more policy PDFs and `requirements_library.csv`. If local inputs are not present, run the setup cell to display an upload control, upload the ZIP, then run that setup cell again.


In [ ]:
# Install project dependencies
%pip install -r requirements.txt


In [ ]:
# Configuration, imports, secrets, and input bootstrap
import os, re, json, time, zipfile, datetime as dt, tempfile, shutil, stat
from pathlib import Path, PurePosixPath
from collections import Counter
import numpy as np, pandas as pd, requests

CONFIG = {
    "input_dir": "sample_policies",
    "requirements_csv": "requirements_library.csv",
    "out_dir": "gap_outputs",
    "top_k": 5,
    "llm_model": "google/gemini-2.5-flash",
    "llm_max_tokens": 1800,
    "org_name": "Group Sustainability & Regulatory Affairs",
}
Path(CONFIG["out_dir"]).mkdir(exist_ok=True)


def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except ImportError:
        return os.environ.get(name)


OPENROUTER_KEY = get_secret("OPENROUTER_API_KEY")
print("LLM refinement:", "ON" if OPENROUTER_KEY else "OFF (rule-based assessment only)")


def extract_input_zip(archive_path):
    global INPUT_TEMP_DIR
    INPUT_TEMP_DIR = tempfile.TemporaryDirectory(prefix="climate_gap_inputs_")
    root = Path(INPUT_TEMP_DIR.name).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = PurePosixPath(member.filename.replace("\\", "/"))
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in uploaded ZIP: {member.filename}")
            mode = member.external_attr >> 16
            if stat.S_ISLNK(mode):
                raise ValueError(f"Symbolic links are not allowed in uploaded ZIPs: {member.filename}")
            destination = (root / Path(*member_path.parts)).resolve()
            destination.relative_to(root)
            if member.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as src, destination.open("wb") as dst:
                    shutil.copyfileobj(src, dst)

    requirements = list(root.rglob("requirements_library.csv"))
    pdfs = list(root.rglob("*.pdf"))
    if len(requirements) != 1:
        raise ValueError("The ZIP must contain exactly one requirements_library.csv")
    if not pdfs:
        raise ValueError("The ZIP must contain at least one policy PDF")
    return root, requirements[0]


def show_local_zip_upload():
    from IPython.display import display
    import ipywidgets as widgets

    uploader = widgets.FileUpload(accept=".zip", multiple=False, description="Upload ZIP")

    def save_upload(change):
        selected = change["new"]
        if isinstance(selected, dict):
            name, item = next(iter(selected.items()))
            content = item["content"]
        elif selected:
            item = selected[0]
            name, content = item["name"], item["content"]
        else:
            return
        if Path(name).suffix.lower() != ".zip":
            raise ValueError("Please select a .zip archive")
        saved_zip = Path(tempfile.gettempdir()) / f"climate_gap_input_{os.getpid()}.zip"
        saved_zip.write_bytes(bytes(content))
        os.environ["CLIMATE_GAP_INPUT_ZIP"] = str(saved_zip)
        print(f"Uploaded {name}. Run this setup cell again to load it.")

    uploader.observe(save_upload, names="value")
    display(uploader)


archive_path = os.environ.get("CLIMATE_GAP_INPUT_ZIP")
if archive_path:
    extracted_root, requirements_path = extract_input_zip(archive_path)
    CONFIG["input_dir"] = str(extracted_root)
    CONFIG["requirements_csv"] = str(requirements_path)
elif not Path(CONFIG["input_dir"]).exists() or not Path(CONFIG["requirements_csv"]).exists():
    try:
        from google.colab import files
    except ImportError:
        show_local_zip_upload()
        raise RuntimeError("Upload the ZIP, then run this setup cell again.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one ZIP containing policy PDFs and requirements_library.csv")
    saved_zip = Path(tempfile.gettempdir()) / f"climate_gap_input_{os.getpid()}.zip"
    saved_zip.write_bytes(uploaded[zip_names[0]])
    extracted_root, requirements_path = extract_input_zip(saved_zip)
    CONFIG["input_dir"] = str(extracted_root)
    CONFIG["requirements_csv"] = str(requirements_path)


### Step 1: Load internal documents and split them into clauses
Each PDF is split at numbered headings. Every clause keeps its document, version, effective date and section number, so each finding can cite the exact place in the source.

In [ ]:
# Cell 3: PDF loading and clause chunking
from pypdf import PdfReader

HEAD = re.compile(r"^(\d+(?:\.\d+)*)\s+([A-Z][^.]{2,80})$")

def doc_label(path):
    return re.sub(r"^\d+_", "", path.stem).replace("_", " ")

def load_chunks(folder):
    chunks, inventory = [], []
    for pdf in sorted(Path(folder).rglob("*.pdf")):
        try:
            pages = [(p.extract_text() or "") for p in PdfReader(str(pdf)).pages]
        except Exception as e:
            print(f"  skip {pdf.name}: {type(e).__name__}"); continue
        text = "\n".join(pages)
        if len(text.strip()) < 50:
            print(f"  WARNING {pdf.name}: no text layer (scanned?). OCR needed."); continue
        m = re.search(r"Version (\S+) \| Effective ([^|]+?) \|", text.replace("\n", " "))
        ver, eff = (m.group(1), m.group(2).strip()) if m else ("n/a", "n/a")
        label, cur, buf = doc_label(pdf), None, []
        def flush():
            body = " ".join(buf).strip()
            if cur and body:
                chunks.append({"doc": label, "file": pdf.name, "section": cur[0], "heading": cur[1], "text": body, "version": ver, "effective": eff})
        for line in text.splitlines():
            line = line.strip()
            if not line: continue
            h = HEAD.match(line)
            if h:
                flush(); cur, buf = (h.group(1), h.group(2)), []
            elif cur:
                buf.append(line)
        flush()
        inventory.append({"Document": label, "Version": ver, "Effective": eff, "Pages": len(pages), "Clauses": sum(1 for c in chunks if c["file"] == pdf.name)})
    return chunks, pd.DataFrame(inventory)

CHUNKS, INVENTORY = load_chunks(CONFIG["input_dir"])
print(f"{len(CHUNKS)} clauses from {len(INVENTORY)} documents")
INVENTORY

### Step 2: Load the external requirements library
Concepts are written as `A|B ; C|D`: a requirement is fully covered only if **every** group (separated by `;`) is evidenced by at least one alternative (separated by `|`, each a regular expression).

In [ ]:
# Cell 4: Requirements library
REQ = pd.read_csv(CONFIG["requirements_csv"]).fillna("")
def parse_groups(s):
    return [[re.compile(a.strip(), re.I) for a in g.split("|") if a.strip()] for g in s.split(";") if g.strip()]
def group_label(r, gi):
    labs = [x.strip() for x in str(r.get("concept_labels", "")).split(";")]
    if gi < len(labs) and labs[gi]: return labs[gi]
    return " / ".join(a.pattern.replace("\\", "").replace(".", " ") for a in r["groups"][gi][:3])
REQS = []
for r in REQ.to_dict("records"):
    r["groups"] = parse_groups(r["concepts"])
    REQS.append(r)
assert REQ.req_id.is_unique, "Duplicate req_id in library"
print(f"{len(REQS)} requirements across {REQ.regime.nunique()} regimes")
REQ.groupby("regime").size().rename("requirements").to_frame()

### Step 3: Embeddings and FAISS index over clauses
Vectors are normalised and searched by inner product (cosine). If the sentence model cannot be downloaded, a TF-IDF fallback keeps the pipeline running; the backend is recorded in the report.

In [ ]:
# Cell 5: Embedder + FAISS
import faiss

class Embedder:
    def __init__(self):
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("all-MiniLM-L6-v2"); self.backend = "sentence-transformers / all-MiniLM-L6-v2"
        except Exception as e:
            print(f"Sentence encoder unavailable ({type(e).__name__}); using TF-IDF fallback")
            self.model, self.backend = None, "TF-IDF fallback (lexical)"
    def fit(self, corpus):
        if self.model is None:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True).fit(corpus)
    def encode(self, texts):
        if self.model is not None:
            v = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        else:
            v = self.vec.transform(texts).toarray(); n = np.linalg.norm(v, axis=1, keepdims=True); v = v / np.where(n == 0, 1, n)
        return np.ascontiguousarray(v, dtype="float32")

chunk_texts = [f"{c['heading']}. {c['text']}" for c in CHUNKS]
req_texts = [f"{r['topic']}. {r['requirement']}" for r in REQS]
emb = Embedder(); emb.fit(chunk_texts + req_texts)
C_VEC, R_VEC = emb.encode(chunk_texts), emb.encode(req_texts)
index = faiss.IndexFlatIP(C_VEC.shape[1]); index.add(C_VEC)
print(f"FAISS: {index.ntotal} clauses, dim={C_VEC.shape[1]}, backend={emb.backend}")

### Step 4: Retrieve evidence and assess coverage
1. FAISS returns the 15 closest clauses; they are re-ranked with a small bonus for each required concept group found in the clause (hybrid semantic + keyword).
2. Concept groups matched in the top clauses set the base status: all matched = *Covered*, at least half = *Partial*, otherwise *Gap*.
3. Evidence sentences are scanned for **aspirational** wording and **negated** wording; each downgrades the status one level.
4. Stale references (for example TCFD without IFRS S2) are flagged.

In [ ]:
# Cell 6: Retrieval + rule-based assessment
WEAK = re.compile(r"\b(will (?:consider|develop|explore|establish|introduce|seek)|under development|is planned|are planned|intends? to|aspires? to|where (?:feasible|practicable)|in due course|not yet|to be (?:introduced|developed|set|defined|established|adopted)|under consideration|being (?:developed|considered))\b", re.I)
NEG = re.compile(r"\b(not (?:yet )?(?:been )?(?:measured|assured|externally assured|implemented|conducted|quantified|included|incorporated|covered|adopted|calculated|performed|disclosed|run)|has not|have not|no formal|no climate|no single|no scenarios?|not systematically)\b", re.I)
ORDER = ["Covered", "Partial", "Gap"]

def sentences(t): return [s.strip() for s in re.split(r"(?<=[.;])\s+", t) if s.strip()]

def retrieve(i, k=None):
    k = k or CONFIG["top_k"]
    scores, ids = index.search(R_VEC[i:i + 1], min(15, index.ntotal))
    ranked = []
    for s, j in zip(scores[0], ids[0]):
        txt = f"{CHUNKS[j]['heading']}. {CHUNKS[j]['text']}"
        bonus = 0.15 * sum(any(rx.search(txt) for rx in g) for g in REQS[i]["groups"])
        ranked.append((float(s) + bonus, int(j)))
    ranked.sort(reverse=True)
    return [j for _, j in ranked[:k]]

def assess(i):
    r = REQS[i]; top = retrieve(i); groups = r["groups"]; G = len(groups)
    clean_ev, flag_ev = [set() for _ in groups], [[] for _ in groups]   # clean chunk ids / flagged (chunk, sentence) per group
    clean_sent = [[] for _ in groups]
    for j in top:
        for s in sentences(CHUNKS[j]["text"]):
            weak, neg = bool(WEAK.search(s)), bool(NEG.search(s))
            for gi, g in enumerate(groups):
                if any(rx.search(s) for rx in g):
                    if weak or neg: flag_ev[gi].append((j, s, weak, neg))
                    else: clean_ev[gi].add(j); clean_sent[gi].append((j, s))
    score, flags = [], []
    for gi in range(G):
        if clean_ev[gi]: score.append(1.0)
        elif flag_ev[gi]:
            score.append(0.0 if gi == 0 else 0.5)          # a flagged-only anchor concept counts as absent
            if any(w for _, _, w, _ in flag_ev[gi]): flags.append("Aspirational wording")
            if any(n for _, _, _, n in flag_ev[gi]): flags.append("Negated wording")
        else: score.append(0.0)
    frac = sum(score) / G
    status = "Covered" if frac == 1 else "Partial" if frac >= 0.5 else "Gap"
    if G > 1 and score[0] == 0: status = "Gap"            # the anchor (first) concept is mandatory
    if status == "Covered" and r.get("same_chunk") in (1, "1") and not set.intersection(*clean_ev):
        status = "Partial"; flags.append("Evidence spread across clauses")
    flags = list(dict.fromkeys(flags))
    alltxt = " ".join(CHUNKS[j]["text"] for j in top)
    if r["stale_pattern"] and re.search(r["stale_pattern"], alltxt, re.I) and not (r["stale_unless"] and re.search(r["stale_unless"], alltxt, re.I)):
        flags.append("Stale reference")
    ev = None
    for gi in range(G):
        if clean_sent[gi]: ev = (1, *clean_sent[gi][0]); break
        if flag_ev[gi]: ev = (1, flag_ev[gi][0][0], flag_ev[gi][0][1]); break
    missing = [group_label(r, gi) for gi in range(G) if score[gi] < 1]
    return {"status": status, "base_status": status, "flags": flags, "top": top, "missing": missing,
            "evidence": ev, "stale_note": r["stale_note"] if "Stale reference" in flags else ""}

ASSESS = [assess(i) for i in range(len(REQS))]
print(Counter(a["status"] for a in ASSESS))

### Step 5 (optional): LLM refinement
With an `OPENROUTER_API_KEY`, every requirement that is not a clean *Covered* is sent to the model with the retrieved clauses and a strict JSON schema. The rule-based result is kept in the register as `baseline_status`, so you can always see where the model changed a call. Without a key this step is skipped.

In [ ]:
# Cell 7: Optional LLM refinement
def call_llm(prompt, retries=3):
    last = None
    for a in range(retries):
        try:
            r = requests.post("https://openrouter.ai/api/v1/chat/completions", timeout=90,
                headers={"Authorization": f"Bearer {OPENROUTER_KEY}", "Content-Type": "application/json"},
                json={"model": CONFIG["llm_model"], "temperature": 0.1, "max_tokens": CONFIG["llm_max_tokens"], "messages": [{"role": "user", "content": prompt}]})
            if r.status_code in (401, 403): raise PermissionError(f"API key rejected (HTTP {r.status_code})")
            r.raise_for_status(); return r.json()["choices"][0]["message"]["content"]
        except PermissionError: raise
        except Exception as e: last = e; time.sleep(2 ** a)
    raise RuntimeError(f"LLM call failed: {last}")

def llm_prompt(r, clauses):
    ctx = "\n".join(f"[{c['doc']} s{c['section']}] {c['text']}" for c in clauses)
    return f"""You are a climate-finance compliance analyst. Judge whether the internal clauses satisfy the external requirement.
Use ONLY the clauses shown. Aspirational or negated wording ("will develop", "not yet") is NOT coverage.
REQUIREMENT ({r['regime']}): {r['requirement']}
CLAUSES:
{ctx}
Return ONLY JSON: {{"status":"Covered|Partial|Gap","rationale":"<60 words","action":"<60 words","proposed_clause":"clause text or empty"}}"""

LLM_OUT = {}
def refine_with_llm():
    n = 0
    for i, a in enumerate(ASSESS):
        if a["status"] == "Covered" and not a["flags"]: continue
        try:
            txt = call_llm(llm_prompt(REQS[i], [CHUNKS[j] for j in a["top"]]))
            d = json.loads(txt[txt.find("{"): txt.rfind("}") + 1])
            if d.get("status") in ORDER:
                LLM_OUT[i] = {k: str(d.get(k, "") or "") for k in ("status", "rationale", "action", "proposed_clause")}; n += 1
        except Exception as e:
            print(f"  {REQS[i]['req_id']}: LLM refinement skipped ({e})");
            if "API key" in str(e): break
    print(f"LLM refined {n} requirements")

if OPENROUTER_KEY: refine_with_llm()
else: print("Skipped (no key).")

### Step 6: Gap register

In [ ]:
# Cell 8: Build the gap register
SHORT = [("IFRS S2", "IFRS S2"), ("CBK Guidance", "CBK Guidance 2021"), ("CBK Green", "CBK Taxonomy/Disclosure 2025"),
         ("Basel", "Basel principles"), ("ICMA", "ICMA GBP"), ("EU", "EU CSRD/ESRS")]
def short(reg): return next((s for p, s in SHORT if reg.startswith(p)), reg)

def priority(obl, status):
    if status == "Covered": return "None"
    strong = obl.startswith(("Mandatory", "Expected"))
    cond = obl.startswith("Conditional")
    if status == "Gap": return "High" if strong else "Medium" if cond else "Low"
    return "Medium" if strong else "Low"

rows = []
for i, (r, a) in enumerate(zip(REQS, ASSESS)):
    ev = a["evidence"]; llm = LLM_OUT.get(i)
    status = llm["status"] if llm else a["status"]
    where = f"{CHUNKS[ev[1]]['doc']} s{CHUNKS[ev[1]]['section']}" if ev else "No supporting clause found"
    quote = (ev[2][:230] + ("..." if len(ev[2]) > 230 else "")) if ev else ""
    finding = quote
    if a["missing"] and a["status"] != "Covered": finding += (" | " if finding else "") + "Missing concepts: " + "; ".join(a["missing"])
    if a["stale_note"]: finding += " | " + a["stale_note"]
    action = (llm["action"] or r["recommendation"]) if llm else r["recommendation"]
    rows.append({"req_id": r["req_id"], "regime": short(r["regime"]), "topic": r["topic"], "requirement": r["requirement"],
        "obligation": r["obligation"], "status": status, "baseline_status": a["status"], "priority": priority(r["obligation"], status),
        "evidence_location": where, "finding": finding, "flags": ", ".join(a["flags"]),
        "recommended_action": "No action needed" if status == "Covered" and not a["flags"] else action,
        "proposed_clause": llm["proposed_clause"] if llm else "", "amend_document": r["owner_doc"], "source_ref": r["source_ref"],
        "assessed_by": "LLM + rules" if llm else "Rules", "owner": "", "due_date": ""})
REGISTER = pd.DataFrame(rows)
PORD = {"High": 0, "Medium": 1, "Low": 2, "None": 3}
REGISTER = REGISTER.sort_values(["priority", "req_id"], key=lambda s: s.map(PORD) if s.name == "priority" else s).reset_index(drop=True)

xlsx = Path(CONFIG["out_dir"]) / "gap_register.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
    REGISTER.to_excel(xw, sheet_name="Gap register", index=False)
    INVENTORY.to_excel(xw, sheet_name="Documents", index=False)
    ws = xw.sheets["Gap register"]
    from openpyxl.styles import PatternFill, Font, Alignment
    widths = {"A": 8, "B": 24, "C": 18, "D": 50, "E": 22, "F": 10, "G": 12, "H": 10, "I": 30, "J": 60, "K": 22, "L": 55, "M": 45, "N": 28, "O": 24, "P": 12, "Q": 14, "R": 14}
    for col, w in widths.items(): ws.column_dimensions[col].width = w
    for c in ws[1]: c.fill = PatternFill("solid", fgColor="14323A"); c.font = Font(bold=True, color="FFFFFF")
    fills = {"Gap": "F4C7C3", "Partial": "FCE8B2", "Covered": "CDE9D6"}
    for row in ws.iter_rows(min_row=2):
        for c in row: c.alignment = Alignment(wrap_text=True, vertical="top")
        row[5].fill = PatternFill("solid", fgColor=fills.get(row[5].value, "FFFFFF"))
    ws.freeze_panes = "B2"; ws.auto_filter.ref = ws.dimensions
REGISTER.to_csv(Path(CONFIG["out_dir"]) / "gap_register.csv", index=False)
print("Saved", xlsx)
print(REGISTER.status.value_counts().to_dict(), "| High priority:", int((REGISTER.priority == "High").sum()))
REGISTER[["req_id", "regime", "topic", "status", "priority", "evidence_location", "flags"]]

### Step 7: Self-checks

In [ ]:
# Cell 9: Checks (the expectations apply to the bundled sample documents only)
assert len(REGISTER) == len(REQS) and REGISTER.req_id.is_unique
assert set(REGISTER.status) <= set(ORDER)
assert all(len(a["top"]) == min(CONFIG["top_k"], len(CHUNKS)) for a in ASSESS)
if CONFIG["input_dir"] == "sample_policies":
    EXPECT = {"R-01": "Covered", "R-02": "Gap", "R-03": "Gap", "R-04": "Gap", "R-05": "Partial", "R-07": "Gap", "R-08": "Partial",
              "R-09": "Partial", "R-10": "Covered", "R-11": "Covered", "R-12": "Gap", "R-13": "Gap", "R-14": "Partial", "R-16": "Gap",
              "R-17": "Gap", "R-18": "Partial", "R-19": "Partial", "R-20": "Covered", "R-23": "Partial", "R-24": "Gap", "R-27": "Gap"}
    got = REGISTER.set_index("req_id").status.to_dict()
    bad = {k: (v, got[k]) for k, v in EXPECT.items() if got[k] != v}
    print("Sample regression check:", "PASS" if not bad else f"DIFFERENCES (expected, got): {bad}")
else:
    print("Custom inputs: regression check skipped")
print("Backend:", emb.backend)

### Step 8: PDF report
Executive summary, coverage by regime, priority gaps with evidence and recommended actions, the full register, the document inventory, and methodology and limitations.

In [ ]:
# Cell 10: PDF report
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from xml.sax.saxutils import escape
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.platypus import BaseDocTemplate, PageTemplate, Frame, Paragraph, Spacer, Table, TableStyle, Image, PageBreak

NAVY, TEAL, GREY, LIGHT = colors.HexColor("#14323a"), colors.HexColor("#1d6b57"), colors.HexColor("#5b6764"), colors.HexColor("#eef2f0")
SC = {"Covered": "#2e8b57", "Partial": "#c98a14", "Gap": "#c0392b"}
PAGE = landscape(A4); W = PAGE[0] - 30 * mm
def clean(t):
    t = str(t or "")
    for a, b in {"\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-"}.items(): t = t.replace(a, b)
    return escape(t.encode("latin-1", "replace").decode("latin-1"))
ss = getSampleStyleSheet()
ST = {"title": ParagraphStyle("t", parent=ss["Title"], fontName="Helvetica-Bold", fontSize=22, leading=26, textColor=NAVY, alignment=0),
      "sub": ParagraphStyle("s", parent=ss["Normal"], fontSize=10.5, leading=14, textColor=GREY),
      "h1": ParagraphStyle("h1", parent=ss["Heading1"], fontName="Helvetica-Bold", fontSize=14, leading=18, textColor=NAVY, spaceBefore=8, spaceAfter=5),
      "b": ParagraphStyle("b", parent=ss["Normal"], fontSize=9.2, leading=12.6),
      "sm": ParagraphStyle("sm", parent=ss["Normal"], fontSize=8, leading=10.5, textColor=GREY),
      "c": ParagraphStyle("c", parent=ss["Normal"], fontSize=7.8, leading=9.8),
      "num": ParagraphStyle("n", parent=ss["Normal"], fontName="Helvetica-Bold", fontSize=22, leading=26)}
def P(t, s="b"): return Paragraph(clean(t), ST[s])
def H(*cells): return [Paragraph(f'<font color="white"><b>{clean(c)}</b></font>', ST["c"]) for c in cells]
def stat(s): return Paragraph(f'<font color="{SC.get(s, "#444")}"><b>{clean(s)}</b></font>', ST["c"])
def tbl(data, widths):
    t = Table(data, colWidths=widths, repeatRows=1)
    t.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "TOP"), ("GRID", (0, 0), (-1, -1), 0.3, colors.HexColor("#c9d1ce")), ("BACKGROUND", (0, 0), (-1, 0), NAVY),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, LIGHT]), ("LEFTPADDING", (0, 0), (-1, -1), 4), ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 3), ("BOTTOMPADDING", (0, 0), (-1, -1), 3)])); return t

def footer(cv, doc):
    cv.saveState(); w, h = PAGE
    cv.setFillColor(NAVY); cv.rect(0, h - 8 * mm, w, 8 * mm, stroke=0, fill=1)
    cv.setFillColor(colors.white); cv.setFont("Helvetica-Bold", 8); cv.drawString(15 * mm, h - 5.4 * mm, "CLIMATE FINANCE POLICY GAP ANALYSIS")
    cv.setFont("Helvetica", 8); cv.drawRightString(w - 15 * mm, h - 5.4 * mm, CONFIG["org_name"])
    cv.setFillColor(GREY); cv.setFont("Helvetica", 7.5)
    cv.drawString(15 * mm, 8 * mm, "Draft for compliance review. Not legal advice. Requirements are paraphrased: verify at the primary source."); cv.drawRightString(w - 15 * mm, 8 * mm, f"Page {doc.page}")
    cv.restoreState()

def chart(path):
    reg = REGISTER.groupby(["regime", "status"]).size().unstack(fill_value=0).reindex(columns=ORDER, fill_value=0)
    reg = reg.loc[[short(x) for x in dict.fromkeys(REQ.regime)]]
    fig, ax = plt.subplots(figsize=(8.6, 0.55 * len(reg) + 1.0), dpi=170); left = np.zeros(len(reg))
    for s in ORDER:
        ax.barh(reg.index, reg[s], left=left, color=SC[s], label=s, height=0.6)
        for y, (v, l) in enumerate(zip(reg[s], left)):
            if v: ax.text(l + v / 2, y, int(v), ha="center", va="center", color="white", fontsize=8, fontweight="bold")
        left += reg[s].values
    ax.invert_yaxis(); ax.set_xlabel("Requirements", fontsize=8); ax.tick_params(labelsize=8); ax.legend(ncol=3, fontsize=8, frameon=False, loc="lower right")
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    fig.tight_layout(); fig.savefig(path); plt.close(fig); return 8.6, 0.55 * len(reg) + 1.0

def build_report():
    path = Path(CONFIG["out_dir"]) / "gap_analysis_report.pdf"; cpath = Path(CONFIG["out_dir"]) / "coverage_by_regime.png"
    cw, ch = chart(cpath)
    doc = BaseDocTemplate(str(path), pagesize=PAGE, leftMargin=15 * mm, rightMargin=15 * mm, topMargin=16 * mm, bottomMargin=15 * mm, title="Climate Finance Policy Gap Analysis", author=CONFIG["org_name"])
    doc.addPageTemplates([PageTemplate(id="p", frames=[Frame(doc.leftMargin, doc.bottomMargin, W, PAGE[1] - 31 * mm, id="f")], onPage=footer)])
    n = REGISTER.status.value_counts().reindex(ORDER, fill_value=0); N = len(REGISTER)
    score = round(100 * (n["Covered"] + 0.5 * n["Partial"]) / N)
    S = [P("Climate Finance Policy Gap Analysis", "title"), P(f"Internal documentation vs external requirements | {dt.date.today():%d %B %Y}", "sub"), Spacer(1, 8)]
    meta = [[P("Documents analysed", "c"), P(f"{len(INVENTORY)} ({len(CHUNKS)} clauses)", "c"), P("Requirements assessed", "c"), P(f"{N} across {REQ.regime.nunique()} regimes", "c"),
             P("Retrieval backend", "c"), P(emb.backend, "c"), P("Assessment", "c"), P("LLM + rules" if LLM_OUT else "Rule-based", "c")]]
    mt = Table(meta, colWidths=[30 * mm, 38 * mm, 32 * mm, 48 * mm, 28 * mm, 40 * mm, 22 * mm, W - 238 * mm]); mt.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.3, colors.HexColor("#c9d1ce")), ("BACKGROUND", (0, 0), (-1, -1), LIGHT), ("VALIGN", (0, 0), (-1, -1), "TOP")]))
    S += [mt, Spacer(1, 8), P("1. Executive summary", "h1")]
    k = [[[Paragraph(f'<font color="{c}">{v}</font>', ST["num"]), P(l, "sm")] for v, l, c in
          [(f"{score}%", "Coverage score (Covered + half of Partial)", "#14323a"), (n["Covered"], "Covered", SC["Covered"]), (n["Partial"], "Partial", SC["Partial"]), (n["Gap"], "Gap", SC["Gap"]), (int((REGISTER.priority == "High").sum()), "High-priority actions", SC["Gap"])]]]
    kt = Table(k, colWidths=[W / 5] * 5); kt.setStyle(TableStyle([("BOX", (0, 0), (-1, -1), 0.4, colors.HexColor("#c9d1ce")), ("INNERGRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#c9d1ce")), ("BACKGROUND", (0, 0), (-1, -1), LIGHT), ("TOPPADDING", (0, 0), (-1, -1), 6), ("BOTTOMPADDING", (0, 0), (-1, -1), 6)]))
    S += [kt, Spacer(1, 6), P("Most important findings", "h1")]
    for _, r in REGISTER[REGISTER.priority == "High"].head(6).iterrows():
        S += [P(f"{r.req_id} | {r.regime} | {r.topic} | {r.status}: {r.recommended_action} (amend: {r.amend_document})", "b"), Spacer(1, 2)]
    S += [Spacer(1, 4), P("Gaps are measured against paraphrased requirements and the wording of the documents provided. A Covered rating means the concepts are evidenced in policy text, not that practice or data are adequate.", "sm"), PageBreak()]
    S += [P("2. Coverage by regime", "h1"), Image(str(cpath), width=190 * mm, height=190 * mm * ch / cw), Spacer(1, 6)]
    rt = [H("Regime", "Requirements", "Covered", "Partial", "Gap", "Coverage")]
    for rg in dict.fromkeys(REQ.regime):
        d = REGISTER[REGISTER.regime == short(rg)]; c = d.status.value_counts().reindex(ORDER, fill_value=0)
        rt.append([P(short(rg), "c"), P(len(d), "c"), P(c["Covered"], "c"), P(c["Partial"], "c"), P(c["Gap"], "c"), P(f"{round(100 * (c['Covered'] + 0.5 * c['Partial']) / len(d))}%", "c")])
    S += [tbl(rt, [70 * mm, 28 * mm, 24 * mm, 24 * mm, 24 * mm, 28 * mm]), PageBreak()]
    S.append(P("3. Priority gaps and recommended actions", "h1"))
    pr = REGISTER[REGISTER.priority.isin(["High", "Medium"])]
    rows = [H("ID", "Regime / topic", "Status", "Priority", "What the documents say (evidence)", "Recommended action", "Amend")]
    for _, r in pr.iterrows():
        ev = r.finding if r.finding else "No supporting clause found."
        rows.append([P(r.req_id, "c"), P(f"{r.regime}: {r.topic}", "c"), stat(r.status), P(r.priority, "c"), P(f"[{r.evidence_location}] {ev}" + (f" ({r['flags']})" if r["flags"] else ""), "c"), P(r.recommended_action, "c"), P(r.amend_document, "c")])
    S += [tbl(rows, [13 * mm, 38 * mm, 15 * mm, 15 * mm, 90 * mm, 66 * mm, W - 237 * mm]), PageBreak()]
    S.append(P("4. Full register", "h1"))
    rows = [H("ID", "Regime", "Topic", "Obligation", "Status", "Priority", "Evidence location", "Flags")]
    for _, r in REGISTER.sort_values("req_id").iterrows():
        rows.append([P(r.req_id, "c"), P(r.regime, "c"), P(r.topic, "c"), P(r.obligation, "c"), stat(r.status), P(r.priority, "c"), P(r.evidence_location, "c"), P(r["flags"], "c")])
    S += [tbl(rows, [13 * mm, 40 * mm, 34 * mm, 42 * mm, 17 * mm, 16 * mm, 55 * mm, W - 217 * mm]), Spacer(1, 8), P("5. Document inventory", "h1")]
    rows = [H("Document", "Version", "Effective", "Pages", "Clauses")] + [[P(r.Document, "c"), P(r.Version, "c"), P(r.Effective, "c"), P(r.Pages, "c"), P(r.Clauses, "c")] for r in INVENTORY.itertuples()]
    S.append(tbl(rows, [100 * mm, 30 * mm, 40 * mm, 20 * mm, 25 * mm]))
    S += [Spacer(1, 8), P("6. Methodology and limitations", "h1")]
    for t in [f"Retrieval: clauses and requirements are embedded ({emb.backend}) and searched with FAISS (cosine); the top {CONFIG['top_k']} clauses per requirement are re-ranked with a keyword bonus for required concepts.",
              "Assessment: coverage is the share of required concept groups evidenced in those clauses. Aspirational wording (for example 'will develop') and negated wording (for example 'not yet implemented') each downgrade the status one level. Stale references are flagged.",
              "Limits: keyword evidence can miss paraphrased coverage and can over-credit loosely related wording; only text PDFs are read; practice, data quality and implementation are not tested. Treat results as a prioritised review list for compliance, not as an audit opinion.",
              "Requirements library: paraphrased summaries with an obligation level (Mandatory, Expected, Conditional, Good practice). Statuses and dates change; confirm at the primary source before approving any policy change."]:
        S += [P(t, "b"), Spacer(1, 3)]
    doc.build(S); return path

REPORT = build_report()
print("PDF written:", REPORT, f"({REPORT.stat().st_size / 1024:.0f} KB)")

In [ ]:
# Cell 11: Download in Colab, or show output paths
try:
    from google.colab import files
    for f in ("gap_analysis_report.pdf", "gap_register.xlsx"): files.download(str(Path(CONFIG["out_dir"]) / f))
except Exception:
    print("Outputs in:", Path(CONFIG["out_dir"]).resolve())

### Next steps with your real documents
1. Replace `sample_policies/` with your PDFs. Each should have numbered headings (`3.2 Title`); if a policy is formatted differently, adjust `HEAD` in Cell 3. Scanned PDFs need OCR first.
2. Edit `requirements_library.csv` against primary sources, add the regimes you report under, and tune the `concepts` column until results look right on 10 to 15 clauses you know well.
3. Add `OPENROUTER_API_KEY` (or an approved internal model endpoint) to refine borderline calls. Check data-handling rules before sending internal text to any external service.
4. Run monthly, keep each `gap_register.xlsx`, and compare runs to show what has been closed.